# Metricas e diagnosticos do baseline

Este notebook concentra as definicoes, resultados e graficos usados para avaliar B0, M1, M2 e M3. Execute antes `03_experimentos_baseline.ipynb` ou `scripts/experimentos/run_baseline_experiments.py` para atualizar os artefatos.

## Metricas principais

| Metrica | Formula / leitura | Melhor valor |
| --- | --- | --- |
| MAE macro | Media, entre canais, de `abs(previsao - resultado)` | Menor |
| RMSE macro | Raiz da media do erro ao quadrado; pune mais erros grandes | Menor |
| Vies macro | Media de `previsao - resultado`; identifica super/subestimativa | Proximo de zero |
| Acuracia de ranking por pares | Proporcao de pares de videos do mesmo canal cuja ordem prevista coincide com a observada | Maior |

As metricas de erro usam `target_log_likes = log(1 + curtidas)`. Nao usamos MAPE: curtidas iguais a zero geram divisao por zero e videos com poucas curtidas distorcem a metrica percentual.

## Comparacoes estatisticas

- **Bootstrap por canal:** reamostra os 12 canais com reposicao para formar um intervalo de confianca da diferenca de MAE.
- **Teste de permutacao pareado:** verifica se a diferenca de erro entre dois modelos poderia surgir ao acaso.
- As comparacoes sao M1 - B0, M2 - M1 e M1 - M3. Valor negativo favorece o primeiro modelo.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "scripts").is_dir() and (p / "src").is_dir())
ARTIFACTS = ROOT / 'artifacts' / 'baseline'
summary = pd.read_csv(ARTIFACTS / 'metrics_summary.csv')
comparisons = json.loads((ARTIFACTS / 'comparison_statistics.json').read_text(encoding='utf-8'))

summary_display = summary.copy()
for column in ['macro_mae_log_likes', 'macro_rmse_log_likes', 'macro_bias_log_likes']:
    summary_display[column] = summary_display[column].map(lambda value: f'{value:.4f}')
summary_display['macro_pairwise_accuracy'] = summary_display['macro_pairwise_accuracy'].map(lambda value: f'{value:.1%}')
display(summary_display)
pd.DataFrame(comparisons).T

## Graficos de diagnostico

A importancia por permutacao mostra quanto o MAE piora quando uma coluna e embaralhada. Ela mede dependencia preditiva do modelo, e nao causalidade.

In [ ]:
figures = ARTIFACTS / 'figures'
for filename in [
    '01_mae_por_modelo.png',
    '02_importancia_por_permutacao_m1.png',
    '03_previsao_versus_resultado_m1.png',
    '04_erro_por_canal_m1_vs_b0.png',
    '05_residuos_por_video_no_tempo_m1.png',
    '06_distribuicao_de_erros_por_tipo_m1.png',
]:
    display(Image(filename=str(figures / filename)))

In [ ]:
importance = pd.read_csv(ARTIFACTS / 'permutation_importance_m1.csv')
importance_display = importance.copy()
for column in ['baseline_macro_mae', 'mean_mae_increase', 'ci_95_low', 'ci_95_high']:
    importance_display[column] = importance_display[column].map(lambda value: f'{value:.4f}')
display(importance_display)